# Lecture 07 — APIs first

> *"The fastest crawler is the one you didn't write because you found the JSON endpoint."*

There is a temptation, when faced with a webpage, to immediately reach for BeautifulSoup. **Don't.** First, ask: *is there a structured-data version of what I want?*

Half the time, the answer is yes. The site has a public API, or a sitemap, or a JSON-LD blob in its `<head>`, or a hidden XHR endpoint that returns clean JSON. Using any of these beats scraping HTML on every axis: faster, more reliable, less brittle, kinder to the server.

This lecture is a tour of the structured-data signals you should look for *before* writing your first selector.

## What you'll be able to do after this lecture

- Find and use a site's `sitemap.xml` to enumerate URLs without crawling.
- Recognize and parse JSON-LD (`<script type="application/ld+json">`).
- Find hidden XHR/JSON endpoints with DevTools.
- Identify GraphQL endpoints and read their schemas.
- Decide quickly which structured source (if any) to use.


## 1. The tier list

When you can pick, prefer the higher tier:

| Tier | Source                              | Why it's best                                  |
|------|-------------------------------------|------------------------------------------------|
| S    | Documented public API               | Stable, designed for programs, ToS-allowed.    |
| A    | Bulk data dump (e.g., Wikipedia)    | One download, no crawl needed.                 |
| A    | RSS/Atom feed                       | Freshness signal designed for machines.        |
| B    | Sitemap (`sitemap.xml`)             | Free URL list, no crawling required.           |
| B    | Hidden XHR/fetch endpoint           | Same data the page uses, often cleaner.        |
| B    | JSON-LD in the page                 | Schema.org-shaped structured data, parser-free.|
| C    | OpenGraph / meta tags               | Limited but stable.                            |
| D    | HTML scraping with selectors        | The fallback. What this whole course teaches.  |

The rest of the lecture is how to find each of these.


## 2. Sitemaps

`sitemap.xml` is a file the site publishes for search engines, listing every URL the site wants indexed. From your perspective: *a free URL inventory*.

Find it three ways:
1. Try `/sitemap.xml` directly.
2. Read `/robots.txt` — it usually has `Sitemap: https://...` lines.
3. Check `/sitemap_index.xml` (sites with many URLs split into multiple files).


In [ ]:
import httpx
from urllib.robotparser import RobotFileParser

# Method 1: from robots.txt
r = httpx.get("https://www.python.org/robots.txt", timeout=10)
print(r.text)


In [ ]:
# Method 2: parse a sitemap directly
import xml.etree.ElementTree as ET
import httpx

r = httpx.get("https://www.python.org/sitemap.xml", timeout=10)
root = ET.fromstring(r.text)

# Sitemap XML has a default namespace; strip it for simpler queries
ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}

# A sitemap index points to other sitemaps; a sitemap has urls
children = list(root)
print("root tag:", root.tag, "-- children:", len(children))
for sub in root.findall(".//sm:loc", ns)[:5]:
    print("  ", sub.text)


For sites with millions of URLs, expect a *sitemap index* (list of sitemap files) rather than a single sitemap. Walk it, gather URLs, deduplicate. You've now built half a crawl plan without a single fetch beyond the sitemap files themselves.

## 3. JSON-LD: structured data inside HTML

Open any well-SEO'd article, product, recipe, or event page and look at the `<head>`. You'll often see:

```html
<script type="application/ld+json">
{
  "@context": "https://schema.org",
  "@type": "NewsArticle",
  "headline": "Apple unveils...",
  "datePublished": "2026-04-26T09:00:00Z",
  "author": {"@type": "Person", "name": "Tim Cook"},
  "image": "https://...",
  "articleBody": "..."
}
</script>
```

That's a single JSON object containing exactly the fields you'd otherwise scrape. **Parse it directly.** Schema.org defines hundreds of `@type`s — Article, Product, Recipe, Event, Person, Organization, etc.

In [ ]:
import json, httpx
from bs4 import BeautifulSoup

URL = "https://www.python.org/"  # not a great example but illustrates the parse
r = httpx.get(URL, timeout=10)
soup = BeautifulSoup(r.text, "lxml")

ld_blocks = []
for tag in soup.find_all("script", type="application/ld+json"):
    try:
        ld_blocks.append(json.loads(tag.string or tag.get_text() or "{}"))
    except json.JSONDecodeError:
        pass

print(f"found {len(ld_blocks)} JSON-LD block(s)")
for blk in ld_blocks[:2]:
    print(json.dumps(blk, indent=2)[:300])


**Real-world tip:** news sites, e-commerce sites, recipe sites, and most modern blogs publish JSON-LD. Always check before writing a selector. The block at the top of an article often has cleaner author/date/headline data than what's visible on the page.

A useful filter: `[b for b in ld_blocks if b.get('@type') == 'NewsArticle']`.

## 4. Hidden XHR / fetch endpoints

This is where DevTools earns its keep. The technique:

1. Open the page in your browser with DevTools → Network tab.
2. Filter by **Fetch/XHR**.
3. Reload the page or interact with it (search, paginate, scroll).
4. Look for requests returning JSON. Click one to see the URL, headers, and response.
5. Right-click → **Copy as cURL**. Paste into a text editor.
6. Convert to Python (manually, or use a [curl-to-python converter](https://curlconverter.com/)).

**What to look for:**
- A path like `/api/...`, `/graphql`, `/v1/...`, `/data/...`, `/_next/data/...`.
- `Content-Type: application/json` in the response.
- A response body that's clean JSON containing the data on the page.

**Pitfall:** sometimes the endpoint requires headers the browser sends automatically (`X-CSRF-Token`, custom auth tokens, `Origin`, `Referer`). Replicate them in `httpx`. If a token is short-lived, you may need to fetch the page first to harvest it, then call the API.

Once you have the endpoint working, you don't need the page anymore. You're calling the same back-end the SPA calls — much cleaner.

In [ ]:
# Toy example: pretend we discovered an API endpoint via DevTools
# (using JSONPlaceholder, a free fake API)
import httpx

resp = httpx.get("https://jsonplaceholder.typicode.com/posts", timeout=10)
posts = resp.json()
print(f"got {len(posts)} posts")
for p in posts[:3]:
    print("  ", p["id"], p["title"])


## 5. GraphQL endpoints

A `POST /graphql` endpoint is a strong signal. The body is a JSON object with `query` (and optionally `variables`):

```json
{
  "query": "query GetUser($id: ID!) { user(id: $id) { name email } }",
  "variables": {"id": "42"}
}
```

How to scrape a GraphQL-backed site:
1. DevTools → Network tab → filter `Fetch/XHR`.
2. Find requests to `/graphql`.
3. Copy the `query` field from the request body.
4. POST the same query (with your own variables) directly with `httpx.post(...)`.

GraphQL also often supports **introspection** — querying the schema itself. If introspection is enabled (often disabled in production), you can map the entire API:

```graphql
{ __schema { types { name fields { name type { name } } } } }
```

Don't introspect against a site that's clearly disabled it for production — that signals "leave us alone.

## 6. RSS / Atom feeds — still alive

For news, blogs, podcasts, and any site with chronological updates, look for:
- `<link rel="alternate" type="application/rss+xml" href="...">` in the `<head>`.
- A `/rss`, `/feed`, `/atom.xml` URL.
- A small RSS icon in the page footer.

Feeds are designed for programmatic consumption, ship in well-defined XML, and almost always carry headline + summary + URL + publish date for free. The Python `feedparser` library does the rest.

In [ ]:
# pip install feedparser
import feedparser

feed = feedparser.parse("https://news.ycombinator.com/rss")
print("feed title:", feed.feed.title)
for entry in feed.entries[:3]:
    print(f"  {entry.published} :: {entry.title}")
    print(f"    {entry.link}")


## 7. OpenGraph and other meta tags

If JSON-LD isn't there, you'll often find a smaller set of meta tags:

```html
<meta property="og:title" content="...">
<meta property="og:description" content="...">
<meta property="og:image" content="...">
<meta property="og:type" content="article">
<meta name="twitter:title" content="...">
```

OpenGraph is what social-media link previews use. It's a fallback structured-data source — more limited than JSON-LD, but more universally present.

In [ ]:
def extract_og(soup):
    og = {}
    for meta in soup.find_all("meta"):
        prop = meta.get("property") or meta.get("name")
        if prop and prop.startswith(("og:", "twitter:")):
            og[prop] = meta.get("content")
    return og


# Try it on the Python homepage
r = httpx.get("https://www.python.org/", timeout=10)
soup = BeautifulSoup(r.text, "lxml")
print(json.dumps(extract_og(soup), indent=2))


## 8. The decision flowchart

```
Need data from a site?
│
├── Public API exists?           ─yes─> Use it. Read the docs. Done.
├── Bulk data dump exists?       ─yes─> Download once. Done.
├── Sitemap exists?              ─yes─> Use it for URL inventory.
├── DevTools shows JSON XHR?     ─yes─> Hit that endpoint with httpx.
├── /graphql endpoint?           ─yes─> Replicate the query.
├── RSS feed?                    ─yes─> feedparser.
├── JSON-LD in page <head>?      ─yes─> json.loads it; you have your fields.
├── OpenGraph tags?              ─yes─> Pull them; supplement with selectors if needed.
└── Otherwise                    ─────> Fall back to BeautifulSoup scraping.
```

Notice how many branches resolve before you ever write a CSS selector. That's the lesson.

## 9. A worked example: Hacker News

Hacker News is famous for being scrape-friendly *because they have an API*. Compare two approaches:

```python
# Approach 1: scrape the homepage HTML
soup = BeautifulSoup(httpx.get("https://news.ycombinator.com/").text, "lxml")
stories = [(row.select_one(".titleline a").get_text(), row.select_one(".titleline a")["href"])
           for row in soup.select("tr.athing")]

# Approach 2: use the API
top_ids = httpx.get("https://hacker-news.firebaseio.com/v0/topstories.json").json()
stories = [httpx.get(f"https://hacker-news.firebaseio.com/v0/item/{i}.json").json() for i in top_ids[:30]]
```

Both work. The API approach is more requests but each is tiny, the data is structured, and HN can't break it by changing their HTML template.

## Recap

- Always look for structured data before scraping HTML. Tier list: API > bulk dump > RSS > sitemap > XHR/JSON > GraphQL > JSON-LD > OpenGraph > HTML.
- Sitemaps and `robots.txt` are free reconnaissance.
- DevTools → Network → Fetch/XHR is how you find hidden APIs. Look for requests returning JSON.
- JSON-LD in `<script type="application/ld+json">` blocks gives you fields without selectors.
- The kind move *and* the smart move are the same here: structured-data first.

## Exercises

1. Pick a news site you read. Find its sitemap. How many articles does it index? Pick three; for each, check if the article page has a JSON-LD block, and if so, what fields it has.
2. Open Twitter/Bluesky/Mastodon in your browser, DevTools → Network → Fetch/XHR. Refresh. Identify the request that loads the timeline. Write down its URL and the keys of its JSON response. (Don't actually hit it — just understand it.)
3. Use the Hacker News API to get the top 30 stories with their points and comment counts. Compare your output to the homepage. Did you save effort? How would maintenance differ over the next year?
4. Find a recipe site that publishes JSON-LD. Extract: ingredients, total time, calories. Now extract the same fields by HTML scraping. Which is shorter? Which would you rather maintain?

## Up next

**Lecture 08** — sometimes the API isn't there, the JSON-LD is empty, and the site has anti-bot protection on top. We'll cover the cat-and-mouse landscape, what your options actually are, and the most under-rated option of all: *don't*.
